# ReefSight — Detecting and Naming Aquarium Fish

*A fine-tuned Faster R-CNN object detector for 13 exotic aquarium fish species*

<div align="center">
<img src="../assets/banner.svg" alt="ReefSight banner" width="800"/>
</div>

Point a camera at an aquarium tank and most people can spot *that there's a fish*. Naming it
is another matter — with 30,000+ known fish species, telling a Moorish Idol from a Butterflyfish
at a glance takes real expertise. ReefSight automates that second step: given a photo, it draws a
box around every fish it finds and labels it with one of 13 species.

This notebook walks through the whole pipeline: loading a YOLO-format dataset, evaluating an
off-the-shelf COCO detector on it (spoiler: it has never heard of fish), then fine-tuning
`fasterrcnn_mobilenet_v3_large_320_fpn` — a lightweight torchvision detector — into a working
fish identifier, entirely on CPU.


### The model: Faster R-CNN

The pipeline starts from a pretrained computer-vision model, checks what it already knows, and
fine-tunes it on the 13 target species.

**Faster R-CNN** works in two stages: a *Region Proposal Network* first scans the image for
hundreds of regions that "look like an object," then a classifier head labels and refines each
region.

Faster R-CNN has a reputation for being heavy. torchvision ships a much lighter variant,
`fasterrcnn_mobilenet_v3_large_320_fpn`: same idea, but with a MobileNetV3 backbone and a
reduced internal resolution (320 px), built for speed — light enough to fine-tune on a laptop
CPU, no GPU required.


### Setup

```bash
uv add torch torchvision kagglehub torchmetrics pandas matplotlib pillow
```

`torchvision` provides the detection models and their pretrained weights, `kagglehub` downloads
the dataset, and `torchmetrics` provides **mAP**, the standard detection metric (used in the
"Going further" section at the end).

`utils.py`, next to this notebook, handles the data plumbing (reading annotation files, building
the photo table) and the plotting helpers, so the notebook itself can stay focused on the
modeling.


In [ ]:
import os
import random

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.transforms import v2 as T
import torchvision.transforms.v2.functional as F
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_320_fpn,
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import kagglehub

from utils import (SPECIES, build_fish_dataframe, collate_fn, compute_map, label_path_for,
                   list_images, plot_boxes, plot_ground_truth_grid, plot_loss_curves,
                   plot_species_counts, read_yolo_boxes)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Part 1 — The dataset

[`fish-dataset`](https://www.kaggle.com/datasets/mahmoodyousaf/fish-dataset) on Kaggle: 8,200+
photos of 13 aquarium fish species, split into train / validation / test, annotated in YOLO
format.


In [ ]:
# Downloads the dataset (~350 MB, once — later runs reuse the local cache)
DATASET_DIR = kagglehub.dataset_download("mahmoodyousaf/fish-dataset", output_dir="data")

print("Dataset downloaded to:", DATASET_DIR)

Each photo has a matching label file (a `.txt` in `data/train/labels`), one line per fish:
`class x_center y_center width height`, the last four numbers expressed as a fraction of the
image width/height (between 0 and 1). The class index indexes into the `SPECIES` list.

Decoding this format is handled by `utils.py`: `read_yolo_boxes` (turns one YOLO line into a
`[x1, y1, x2, y2]` pixel rectangle) and `build_fish_dataframe` (walks the three splits and builds
a one-row-per-photo table, with each photo's dominant species).


In [ ]:
print(f"{len(SPECIES)} species: {', '.join(SPECIES)}")

fish_df = build_fish_dataframe(DATASET_DIR)
display(fish_df.head())
print(f"{len(fish_df)} labelled photos in total")

plot_species_counts(fish_df, split="train")

The distribution is far from uniform: Gourami has almost 10x more photos than YellowCichlid.
Worth keeping in mind for later.

A table of coordinates doesn't say much on its own — let's draw the real boxes on six random
training photos instead.


In [ ]:
sample_paths = fish_df[fish_df["split"] == "train"]["image_path"].sample(n=6, random_state=SEED).tolist()
plot_ground_truth_grid(sample_paths)

# Part 2 — The pretrained detector, as-is

`fasterrcnn_mobilenet_v3_large_320_fpn` is pretrained on **COCO**, a reference object-detection
dataset (people, cars, pets, furniture...) with 91 categories. Let's load it and see how it does
on fish, unmodified.


In [ ]:
weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT

coco_model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights).eval()  # pretrained detector, in eval mode
coco_categories = weights.meta["categories"]                                # the 91 COCO categories

print(f"Faster R-CNN (MobileNetV3) loaded: {sum(p.numel() for p in coco_model.parameters()):,} parameters")
print("COCO categories containing 'fish':", [c for c in coco_categories if "fish" in c.lower()] or "NONE")

No surprise: COCO has no fish category at all. Let's see what the model guesses anyway.

### Running inference

`predict` takes a model and an image path, runs the detector, and keeps only the boxes whose
confidence score is above `score_thresh`. This is the core inference routine for detection —
reused as-is later with the fine-tuned model.

1. load the image with `read_image(image_path)[:3]` (the `[:3]` drops a possible alpha channel),
   convert it to normalized float with `F.to_dtype(image, torch.float32, scale=True)`, then add a
   batch dimension with `.unsqueeze(0)`
2. run that batch through the model (inside `with torch.no_grad():`); a single image is sent, so
   keep the first (and only) result
3. build a boolean mask of detections confident enough (`scores > score_thresh`)
4. filter `boxes`, `labels` (mapped through `names`) and `scores` with that same mask


In [ ]:
def predict(model, image_path, names, score_thresh=0.5):
    """
    Run a detection model on an image and keep the detections above score_thresh.

    Arguments:
    model -- a torchvision detection model, in eval mode
    image_path -- path to an image file, of any resolution
    names -- list of category names, indexed by the label indices the model outputs
    score_thresh -- minimum confidence score to keep a detection

    Returns:
    boxes -- list of [x1, y1, x2, y2] pixel boxes
    labels -- list of category names, same order as boxes
    scores -- list of confidence scores, same order as boxes
    """
    image = F.to_dtype(read_image(image_path)[:3], torch.float32, scale=True)
    batch = image.unsqueeze(0)
    with torch.no_grad():
        output = model(batch)[0]

    keep = output["scores"] > score_thresh
    boxes = output["boxes"][keep].tolist()
    labels = [names[i] for i in output["labels"][keep].tolist()]
    scores = output["scores"][keep].tolist()

    return boxes, labels, scores


goldfish_path = os.path.join(DATASET_DIR, "train", "images",
                             "batch_1_34_goldfish_16_jpg.rf.631bf82a6ed6e4ea66637d3ca6b6dae7.jpg")
boxes, labels, scores = predict(coco_model, goldfish_path, coco_categories)
plot_boxes(Image.open(goldfish_path).convert("RGB"), boxes, labels, scores, box_color="orange",
           title="Raw COCO detector - true species: GoldFish")

print(f"{len(boxes)} detection(s): {list(zip(labels, [round(s, 2) for s in scores]))}")
assert isinstance(boxes, list) and isinstance(labels, list) and isinstance(scores, list), \
    "boxes, labels and scores must be lists"
assert len(boxes) == len(labels) == len(scores), "the three lists must have the same length"
print("Sanity check passed.")

Telling: on a goldfish photographed up close, the model sees... a "bird"! A second photo to
confirm:


In [ ]:
clownfish_path = os.path.join(DATASET_DIR, "train", "images",
                              "CLOWN-PERCULA-CLASSIC-AQUACULTURED-PAIR-Amphiprion-percula_jpg.rf.2d0bece80bf5100491a22972fee01a23.jpg")

boxes, labels, scores = predict(coco_model, clownfish_path, coco_categories, score_thresh=0.3)
plot_boxes(Image.open(clownfish_path).convert("RGB"), boxes, labels, scores, box_color="orange",
           title="Raw COCO detector - true species: ClownFish")

Same verdict for the clownfish. COCO simply never saw fish, as a concept or as species. The
model still localizes the interesting region of the image reasonably well — the Region Proposal
Network knows how to spot "an object" — it only gets the label wrong. Encouraging: half the job
(localizing) is already there, the rest (classifying correctly) is what fine-tuning is for.

Bonus: this detector accepts images of **any size** — it resizes internally and rescales the
output boxes back to the original image's coordinate system. A raw 4032x3024 phone photo can be
fed to it directly, no manual preprocessing needed.


# Part 3 — Feeding data to the model

We have the photos. We have the boxes, already decoded into pixels by `read_yolo_boxes`. What's
left is **serving** them to the model — less trivial than it sounds.

Training will ask the same question thousands of times: *give me sample number 3812*. Every time,
three things need to happen:

1. **open the photo** — images aren't all loaded into memory at once; they're read one at a time,
   on demand
2. **turn the boxes into tensors** — they exist as numbers in a text file, but the model only
   accepts PyTorch tensors
3. **package everything** into the exact structure torchvision expects

This is exactly what a PyTorch **`Dataset`** is for. Careful with a common mix-up: this isn't a
raw data store, it's a recipe — a `__getitem__(i)` method that builds sample `i` on demand.


In [ ]:
basic_transform = T.Compose([T.ToDtype(torch.float32, scale=True)])  # no augmentation for now


class FishDetectionDataset(Dataset):
    def __init__(self, image_paths, transforms=None):
        self.image_paths = image_paths
        self.transforms = transforms

    def __len__(self):
        """How many samples in total? The DataLoader needs to know."""
        return len(self.image_paths)

    def __getitem__(self, i):
        """Build sample number i: an image, and everything known about it."""
        image_path = self.image_paths[i]
        image = read_image(image_path)[:3]                                    # the photo, as a (3, H, W) tensor
        H, W = image.shape[-2:]
        boxes, class_indices = read_yolo_boxes(label_path_for(image_path), W, H)

        # The boxes become a tensor of shape (number of fish, 4)
        boxes_t = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)

        # +1 on every species index. torchvision reserves class 0 for "background"
        # (no object), so our 13 species occupy indices 1 to 13.
        # dtype=torch.int64 is required: on a photo with no fish, the list is empty and
        # torch.tensor([]) would default to float, which the model rejects.
        labels_t = torch.tensor([c + 1 for c in class_indices], dtype=torch.int64)

        # Wrapped in tv_tensors: this is what lets a rotation or a flip later move the
        # boxes IN SYNC with the image.
        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes_t, format="XYXY", canvas_size=(H, W)),
            "labels": labels_t,
        }
        image = tv_tensors.Image(image)

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target


# A look at what the recipe produces, on a photo we already know
demo_image, demo_target = FishDetectionDataset([clownfish_path], transforms=basic_transform)[0]

print(f"image : tensor {tuple(demo_image.shape)}, values between {demo_image.min():.2f} and {demo_image.max():.2f}")
print(f"boxes : {demo_target['boxes'].shape[0]} fish, coordinates {demo_target['boxes'].tolist()}")
print(f"labels: {demo_target['labels'].tolist()}  ->  {[SPECIES[i - 1] for i in demo_target['labels'].tolist()]}")

This is what **one** sample looks like: an image, and a `target` dictionary holding all its
boxes and species. This pair, and nothing else, is what the model receives thousands of times
over.

One last detail before batching. From one sample to the next, images don't share the same size
and don't contain the same number of fish: they can't be stacked into a single tensor the way
classification batches are. That's the role of `collate_fn` (in `utils.py`): it simply keeps
**lists**. The detector handles a list of differently-sized images natively.

`MAX_TRAIN_IMAGES` caps the number of training photos to stay fast on a laptop: with 1,000 photos
and 3 epochs, expect roughly 5 minutes on a recent CPU. Raise these values if your machine can
take it, or move to Google Colab.


In [ ]:
MAX_TRAIN_IMAGES = 1000
MAX_VAL_IMAGES = 300
BATCH_SIZE = 4        # lower this if you run out of memory

all_train, all_val = list_images(DATASET_DIR, "train"), list_images(DATASET_DIR, "valid")
train_images = random.sample(all_train, min(MAX_TRAIN_IMAGES, len(all_train)))
val_images = random.sample(all_val, min(MAX_VAL_IMAGES, len(all_val)))

train_loader = DataLoader(FishDetectionDataset(train_images, transforms=basic_transform),
                          batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(FishDetectionDataset(val_images, transforms=basic_transform),
                        batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

images, targets = next(iter(train_loader))
print(f"One batch = {len(images)} images of sizes {[tuple(im.shape[-2:]) for im in images]}")
print(f"            and {[len(t['labels']) for t in targets]} fish respectively")
print()
print(f"Training  : {len(train_images)} photos, {len(train_loader)} batches of {BATCH_SIZE}")
print(f"Validation: {len(val_images)} photos, {len(val_loader)} batches of {BATCH_SIZE}")

# Part 4 — Fine-tuning

Data is ready — on to the final part: fine-tuning the pretrained model on the 13 species.

The approach: freeze the body of the network (`model.backbone`), and only train the box
classification head (`model.roi_heads.box_predictor`).

### Adapting the model

`build_model`:
1. freezes every parameter of `model.backbone` (`requires_grad = False`)
2. replaces `model.roi_heads.box_predictor` with a new
   `FastRCNNPredictor(in_features, num_classes_with_background)`, where
   `in_features = model.roi_heads.box_predictor.cls_score.in_features` (the input size of the
   current head)

`num_classes_with_background` is 14 here (13 species + background).


In [ ]:
def build_model(num_classes_with_background):
    """
    Load a pretrained Faster R-CNN (MobileNetV3 backbone) and adapt it to a new number of classes.

    Arguments:
    num_classes_with_background -- number of species to detect, PLUS the background class

    Returns:
    model -- the adapted model, with a frozen backbone and a new box-classification head
    """
    model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT)

    for p in model.backbone.parameters():
        p.requires_grad = False                            # the backbone already knows how to "see"
    in_features = model.roi_heads.box_predictor.cls_score.in_features   # input size of the old head
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes_with_background)

    return model


model = build_model(len(SPECIES) + 1)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{n_trainable:,} trainable parameters out of {sum(p.numel() for p in model.parameters()):,} total")

assert model.roi_heads.box_predictor.cls_score.out_features == 14, "expected 14 outputs (13 species + background)"
assert all(not p.requires_grad for p in model.backbone.parameters()), "model.backbone should be frozen"
print("Sanity check passed.")

### The training loop

Unlike a classification task, in training mode (`model.train()`), calling `model(images, targets)`
does **not** return predictions — it returns a dictionary of 4 losses (`loss_classifier`,
`loss_box_reg`, `loss_objectness`, `loss_rpn_box_reg`), one per sub-part of the network. They're
summed into a single total loss to backpropagate.

1. `loss_dict = model(images, targets)` (both already lists, thanks to `collate_fn`)
2. total loss: `loss = sum(loss_dict.values())`
3. zero the gradients with `optimizer.zero_grad()`, backpropagate with `loss.backward()`, then
   update the weights with `optimizer.step()`

Expect roughly 10-30 minutes of compute, depending on your machine.


In [ ]:
NUM_EPOCHS = 3  # lower this if it's too slow
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)

loss_names = ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]
loss_history = {name: [] for name in loss_names}

for epoch in range(NUM_EPOCHS):
    model.train()
    running = {name: 0.0 for name in loss_names}
    n_batches = 0
    for images, targets in train_loader:
        images, targets = list(images), list(targets)

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for name in loss_names:
            running[name] += loss_dict[name].item()
        n_batches += 1

    for name in loss_names:
        loss_history[name].append(running[name] / n_batches)
    print(f"epoch {epoch + 1}/{NUM_EPOCHS}  total loss {sum(loss_history[name][-1] for name in loss_names):.3f}")

assert sum(loss_history[name][-1] for name in loss_names) < sum(loss_history[name][0] for name in loss_names), \
    "the total loss should decrease between the first and last epoch"
print("Sanity check passed.")

In [ ]:
plot_loss_curves(loss_history, NUM_EPOCHS)

`loss_classifier` and `loss_box_reg` (is it the right fish, and is the box well placed) usually
drop the most, since they're the only parts of the network actually concerned with OUR 13
species. `loss_objectness` and `loss_rpn_box_reg` come from the Region Proposal Network, which
already knew how to spot "an object" from its COCO pretraining, and so has little new to learn.


# Part 5 — The fine-tuned detector at work

Using the fine-tuned model just means calling the `predict` function from Part 2 with the
fine-tuned model, and the list of names matching ITS outputs.

**Mind the offset**: in Part 3, the `Dataset` shifted species indices by +1 to leave class 0 for
background. The names list passed to `predict` starts with background, then the 13 species:
output label `1` correctly points at `SPECIES[0]`.

`names = ["background"] + SPECIES`, then `predict(model, image_path, names, score_thresh)`.
`model.eval()` first — no more gradients, only predictions.


In [ ]:
model.eval()


def detect_and_label_fish(image_path, score_thresh=0.5):
    """
    Detect and identify every fish in a photo, using the fine-tuned model.

    Arguments:
    image_path -- path to an image file, of any resolution
    score_thresh -- minimum confidence score to keep a detection

    Returns:
    boxes -- list of [x1, y1, x2, y2] pixel boxes
    labels -- list of species names, same order as boxes
    scores -- list of confidence scores, same order as boxes
    """
    names = ["background"] + SPECIES   # class 0 is background, our species follow: label 1 -> SPECIES[0]
    boxes, labels, scores = predict(model, image_path, names, score_thresh)

    return boxes, labels, scores


boxes, labels, scores = detect_and_label_fish(clownfish_path)
plot_boxes(Image.open(clownfish_path).convert("RGB"), boxes, labels, scores, title="ReefSight identification")

print(f"{len(boxes)} detection(s): {list(zip(labels, [round(s, 2) for s in scores]))}")
assert all(label in SPECIES for label in labels), \
    f"every label should be a species (watch the background offset), got {labels}"
print("Sanity check passed.")

Try it on your own photos: drop an image into the project folder and call
`detect_and_label_fish("my_file.jpg")`. Any resolution works, including a raw phone photo — the
model handles resizing internally.

**Note**: the model only knows the 13 species it was trained on. On a fish of another species, it
will likely still detect *something* (an object, a shape), but will tag it with the closest of
the 13 species — it never says "I don't recognize this species."


---

# Summary

Key takeaways from this project:

1. **Detecting isn't classifying.** An aquarium photo isn't a single label — every fish present
   raises two questions: where is it, and what is it?
2. **The raw pretrained detector** (COCO) had no fish category at all: it confused our species
   with birds, while still correctly localizing the interesting region of the image.
3. **`Dataset` and `DataLoader`** are the belt between files and network: a recipe that builds
   sample `i` on demand, and a dispenser that calls it in a loop. You'll see this pair in every
   PyTorch project.
4. **Fine-tuning**, by freezing the backbone and only replacing the box classification head, was
   enough to get a working detector in a few minutes of CPU compute, with no data augmentation at
   all.

Some limitations: no data augmentation, an imbalanced dataset, a deliberately capped
`MAX_TRAIN_IMAGES`, and no way to flag a species as "unknown." The rarer species in the dataset
also give the network very few box examples to learn from, and suffer more than the common ones.

### Going further

- **measure** the detector instead of eyeballing it: `compute_map(model, val_loader)` (provided
  in `utils.py`) computes **mAP**, the standard detection metric — it compares predicted boxes to
  ground truth at several overlap thresholds and averages over the 13 species
- train on the full 6,800 photos with more epochs (ideally on Google Colab with a GPU), and
  compare the resulting mAP
- try `fasterrcnn_resnet50_fpn`, the "large" Faster R-CNN variant (slower, often more accurate),
  and compare
- the dataset is natively in YOLO format: the `ultralytics` library trains an actual YOLO model in
  a few lines (`yolo detect train data=data.yaml model=yolo11n.pt`) — an interesting baseline to
  compare against this Faster R-CNN
- add box-aware data augmentation (`torchvision.transforms.v2` + `tv_tensors`:
  `RandomHorizontalFlip`, `RandomPhotometricDistort`, `RandomZoomOut`...) and check whether mAP
  actually improves on this dataset
- build a real app with `streamlit`, taking a phone camera feed as direct input
